In [2]:
import pandas as pd
from pathlib import Path

dossier_processed = Path("../data/processed")

df_inflation = pd.read_csv(dossier_processed / "inflation_senegal_features.csv")
df_taux_change = pd.read_csv(dossier_processed / "taux_change_uemoa_features.csv")
df_pib = pd.read_csv(dossier_processed / "pib_senegal_features.csv")

# On convertit la colonne period en vraie date, utile pour les modèles de série temporelle
df_inflation['period'] = pd.to_datetime(df_inflation['period'])
df_taux_change['period'] = pd.to_datetime(df_taux_change['period'])
df_pib['period'] = pd.to_datetime(df_pib['period'])

print(df_inflation.shape, df_taux_change.shape, df_pib.shape)

(27, 7) (65, 7) (65, 7)


In [3]:
# Modèle naïf : on prédit que la prochaine valeur = dernière valeur connue
# On calcule son erreur pour avoir un point de comparaison

# On sépare : tout sauf la dernière ligne pour "entraîner" (ici rien à entraîner, mais on garde la logique)
# et la dernière ligne pour tester la prédiction
train_inflation = df_inflation.iloc[:-1]
test_inflation = df_inflation.iloc[-1:]

# Prédiction naïve : la valeur prédite = dernière valeur connue avant le test
prediction_naive = train_inflation['value'].iloc[-1]
valeur_reelle = test_inflation['value'].iloc[0]

erreur_naive = abs(prediction_naive - valeur_reelle)

print("Dernière valeur connue (prédiction) :", prediction_naive)
print("Valeur réelle (2024) :", valeur_reelle)
print("Erreur absolue du modèle naïf :", erreur_naive)

Dernière valeur connue (prédiction) : 5.9
Valeur réelle (2024) : 0.8
Erreur absolue du modèle naïf : 5.1000000000000005


In [4]:
# Validation glissante : on teste le modèle naïf sur plusieurs années successives,
# pas seulement la dernière, pour avoir une mesure d'erreur plus fiable

erreurs = []

# On teste sur les 10 dernières années disponibles (à ajuster si trop ou trop peu de données)
for i in range(10, 0, -1):
    train = df_inflation.iloc[:-i]
    valeur_reelle = df_inflation['value'].iloc[-i]
    prediction_naive = train['value'].iloc[-1]

    erreur = abs(prediction_naive - valeur_reelle)
    erreurs.append(erreur)

# Erreur absolue moyenne (MAE) du modèle naïf sur ces 10 années
mae_naive = sum(erreurs) / len(erreurs)
print("Erreurs sur les 10 dernières années :", [round(e, 2) for e in erreurs])
print("Erreur absolue moyenne (MAE) du modèle naïf :", round(mae_naive, 2))

Erreurs sur les 10 dernières années : [np.float64(1.3), np.float64(0.6), np.float64(1.4), np.float64(1.7), np.float64(0.5), np.float64(1.5), np.float64(0.3), np.float64(7.5), np.float64(3.8), np.float64(5.1)]
Erreur absolue moyenne (MAE) du modèle naïf : 2.37


In [5]:
from statsmodels.tsa.statespace.sarimax import SARIMAX
import warnings
warnings.filterwarnings("ignore")  # SARIMA génère des avertissements techniques sans gravité

# On entraîne un SARIMA simple sur toute la série d'inflation
modele = SARIMAX(df_inflation['value'], order=(1, 1, 1))
resultat = modele.fit(disp=False)

print(resultat.summary())

                               SARIMAX Results                                
Dep. Variable:                  value   No. Observations:                   27
Model:               SARIMAX(1, 1, 1)   Log Likelihood                 -60.471
Date:                Tue, 28 Jul 2026   AIC                            126.942
Time:                        02:10:19   BIC                            130.717
Sample:                             0   HQIC                           128.029
                                 - 27                                         
Covariance Type:                  opg                                         
                 coef    std err          z      P>|z|      [0.025      0.975]
------------------------------------------------------------------------------
ar.L1          0.3271      0.400      0.818      0.413      -0.457       1.111
ma.L1         -0.9996     58.759     -0.017      0.986    -116.166     114.167
sigma2         5.5401    324.194      0.017      0.9

In [6]:
# Validation glissante pour SARIMA : on répète l'entraînement à chaque étape,
# en ajoutant à chaque fois une année de plus, pour rester réaliste (pas de triche avec le futur)

erreurs_sarima = []

for i in range(10, 0, -1):
    train = df_inflation['value'].iloc[:-i]
    valeur_reelle = df_inflation['value'].iloc[-i]

    modele = SARIMAX(train, order=(1, 1, 1))
    resultat = modele.fit(disp=False)

    prediction = resultat.forecast(steps=1).iloc[0]
    erreur = abs(prediction - valeur_reelle)
    erreurs_sarima.append(erreur)

mae_sarima = sum(erreurs_sarima) / len(erreurs_sarima)
print("Erreurs SARIMA sur les 10 dernières années :", [round(e, 2) for e in erreurs_sarima])
print("MAE SARIMA :", round(mae_sarima, 2))
print("MAE modèle naïf (référence) :", 2.37)

Erreurs SARIMA sur les 10 dernières années : [np.float64(0.82), np.float64(0.4), np.float64(0.89), np.float64(1.18), np.float64(0.26), np.float64(1.15), np.float64(0.49), np.float64(8.03), np.float64(1.5), np.float64(2.81)]
MAE SARIMA : 1.75
MAE modèle naïf (référence) : 2.37


In [7]:
# Modèle naïf sur le taux de change
erreurs_naive_tc = []
for i in range(10, 0, -1):
    train = df_taux_change['value'].iloc[:-i]
    valeur_reelle = df_taux_change['value'].iloc[-i]
    prediction_naive = train.iloc[-1]
    erreurs_naive_tc.append(abs(prediction_naive - valeur_reelle))

mae_naive_tc = sum(erreurs_naive_tc) / len(erreurs_naive_tc)

# Modèle SARIMA sur le taux de change
erreurs_sarima_tc = []
for i in range(10, 0, -1):
    train = df_taux_change['value'].iloc[:-i]
    valeur_reelle = df_taux_change['value'].iloc[-i]

    modele = SARIMAX(train, order=(1, 1, 1))
    resultat = modele.fit(disp=False)
    prediction = resultat.forecast(steps=1).iloc[0]
    erreurs_sarima_tc.append(abs(prediction - valeur_reelle))

mae_sarima_tc = sum(erreurs_sarima_tc) / len(erreurs_sarima_tc)

print("MAE naïf (taux de change) :", round(mae_naive_tc, 2))
print("MAE SARIMA (taux de change) :", round(mae_sarima_tc, 2))

MAE naïf (taux de change) : 28.47
MAE SARIMA (taux de change) : 31.26


In [8]:
# Modèle naïf sur le PIB
erreurs_naive_pib = []
for i in range(10, 0, -1):
    train = df_pib['value'].iloc[:-i]
    valeur_reelle = df_pib['value'].iloc[-i]
    prediction_naive = train.iloc[-1]
    erreurs_naive_pib.append(abs(prediction_naive - valeur_reelle))

mae_naive_pib = sum(erreurs_naive_pib) / len(erreurs_naive_pib)

# Modèle SARIMA sur le PIB
erreurs_sarima_pib = []
for i in range(10, 0, -1):
    train = df_pib['value'].iloc[:-i]
    valeur_reelle = df_pib['value'].iloc[-i]

    modele = SARIMAX(train, order=(1, 1, 1))
    resultat = modele.fit(disp=False)
    prediction = resultat.forecast(steps=1).iloc[0]
    erreurs_sarima_pib.append(abs(prediction - valeur_reelle))

mae_sarima_pib = sum(erreurs_sarima_pib) / len(erreurs_sarima_pib)

print("MAE naïf (PIB) :", round(mae_naive_pib, 2))
print("MAE SARIMA (PIB) :", round(mae_sarima_pib, 2))

MAE naïf (PIB) : 1059.06
MAE SARIMA (PIB) : 379.1


In [9]:
import itertools

def meilleur_sarima(serie, ordres_a_tester, nb_annees_test=10):
    """Teste plusieurs combinaisons d'ordres SARIMA et retourne la meilleure selon le MAE en validation glissante."""
    resultats = []

    for ordre in ordres_a_tester:
        erreurs = []
        try:
            for i in range(nb_annees_test, 0, -1):
                train = serie.iloc[:-i]
                valeur_reelle = serie.iloc[-i]

                modele = SARIMAX(train, order=ordre)
                resultat = modele.fit(disp=False)
                prediction = resultat.forecast(steps=1).iloc[0]
                erreurs.append(abs(prediction - valeur_reelle))

            mae = sum(erreurs) / len(erreurs)
            resultats.append((ordre, mae))
        except Exception:
            # Certaines combinaisons peuvent échouer à converger, on les ignore simplement
            continue

    resultats.sort(key=lambda x: x[1])
    return resultats

# On teste toutes les combinaisons simples de p, d, q entre 0 et 2
ordres_a_tester = list(itertools.product(range(3), range(2), range(3)))
print("Nombre de combinaisons à tester :", len(ordres_a_tester))

Nombre de combinaisons à tester : 18


In [10]:
# Recherche du meilleur SARIMA pour l'inflation (peut prendre 30s à 1-2 min)
resultats_inflation = meilleur_sarima(df_inflation['value'], ordres_a_tester)

print("Top 5 des meilleures combinaisons (ordre, MAE) :")
for ordre, mae in resultats_inflation[:5]:
    print(ordre, "->", round(mae, 3))

print("\nPour rappel, MAE naïf :", 2.37, "| MAE SARIMA(1,1,1) initial :", 1.75)

Top 5 des meilleures combinaisons (ordre, MAE) :
(1, 1, 2) -> 1.418
(2, 0, 2) -> 1.427
(2, 1, 2) -> 1.548
(2, 1, 1) -> 1.67
(0, 1, 2) -> 1.75

Pour rappel, MAE naïf : 2.37 | MAE SARIMA(1,1,1) initial : 1.75


In [11]:
# Recherche du meilleur SARIMA pour le taux de change
resultats_tc = meilleur_sarima(df_taux_change['value'], ordres_a_tester)
print("Top 3 taux de change :")
for ordre, mae in resultats_tc[:3]:
    print(ordre, "->", round(mae, 3))
print("Rappel MAE naïf :", 28.47, "\n")

# Recherche du meilleur SARIMA pour le PIB
resultats_pib = meilleur_sarima(df_pib['value'], ordres_a_tester)
print("Top 3 PIB :")
for ordre, mae in resultats_pib[:3]:
    print(ordre, "->", round(mae, 3))
print("Rappel MAE naïf :", 1059.06)

Top 3 taux de change :
(1, 0, 0) -> 28.452
(0, 1, 0) -> 28.467
(2, 0, 1) -> 29.027
Rappel MAE naïf : 28.47 

Top 3 PIB :
(2, 1, 0) -> 361.606
(1, 1, 1) -> 379.103
(2, 1, 1) -> 387.606
Rappel MAE naïf : 1059.06
